# Suite DIA — The Python, Matlab, Excel and Latex dialects

Each dialect models a language's expressions: its kinds, its vocabulary and the domains of its values, `render`
to its source text, an evaluator with the language's own rules, and a scope that resolves names, imports and
references as the language does. Python evaluates by Python's rules (modeled over JavaScript values here);
MATLAB is two-valued over doubles, logicals and strings, with functions from the path and packages; Excel coerces,
computes with error values, and reads cells from workbooks. Latex writes notation, and has no evaluator. What every dialect shares is suite FRM; the Basic
dialect is suites EXP and EVL. Reading Python source (`parse`) and NumPy are the Python implementation's only.

In [ ]:
import * as Excel from "@mbse/expressions/Dialects/Excel";
import * as Latex from "@mbse/expressions/Dialects/Latex";
import * as Matlab from "@mbse/expressions/Dialects/Matlab";
import * as Python from "@mbse/expressions/Dialects/Python";
import { Domains as D, Errors as FE, Terms as F } from "@mbse/expressions/Framework";
import { Errors, Proxies, Repr, Schemas as S } from "@mbse/schemas/Framework";

const { AttributeError, KeyError, ValueError } = Errors;
const { ImportError, NameError, OverflowError, ZeroDivisionError } = FE;
import { assert, raises, same, text } from "./support.js";

const [P, M, X] = [Python.Expressions, Matlab.Expressions, Excel.Expressions];
const [PV, MV, XV] = [Python.Evaluators, Matlab.Evaluators, Excel.Evaluators];
const [PD, MD, XD] = [Python.Domains, Matlab.Domains, Excel.Domains];
const kindOf = (dialect: F.Declared, tag: string) => dialect.kinds().get(tag) as any;

const Person = new S.OfObject.Builder().ref().properties(text("name"), text("age", BigInt), text("email")).create();
Proxies.register("Person", Person);
const B = (Proxies.Builders as any).Person;
const ann = B().name("Ann").age(30n).email("ann@example.com").create();
const bob = B().name("Bob").age(70n).create(); // no email

## DIA-01 · Python: expressions as `ast` has them, with their imports; rendered and parsed as source

In [ ]:
{
  const [t, a, b, c, d, x, y, z] = [..."tabcdxyz"].map((n) => P.name(n));
  for (const [built, source] of [ // written with constructors, as source
    [P.import_("numpy", P.importfrom("numpy.ma", "getmaskarray", P.call(
      "np.logical_and", P.call("np.greater_equal", P.subscript(t, "age"), 18n),
      P.call("np.logical_not", P.call("getmaskarray", P.subscript(t, "email"))))), "np"),
    "import numpy as np\nfrom numpy.ma import getmaskarray\n"
      + "np.logical_and(np.greater_equal(t['age'], 18), np.logical_not(getmaskarray(t['email'])))"],
    [P.let_("a", t, P.boolop("and", P.compare(">", P.attribute(a, "b"), 1n), P.unaryop("not", P.attribute(a, "c")))),
      "(lambda a: a.b > 1 and not a.c)(t)"],
    [P.ifexp(P.compare("<", a, b), x, P.unaryop("-", P.binop("**", y, 2n))), "x if a < b else -y ** 2"],
    [P.ifexp(b, P.ifexp(a, x, y), z), "(x if a else y) if b else z"],
    [P.ifexp(a, x, P.ifexp(b, y, z)), "x if a else y if b else z"],
    [P.binop("-", P.binop("-", P.binop("**", 2n, P.unaryop("-", 1n)), P.binop("-", a, b)), c), "2 ** -1 - (a - b) - c"],
    [P.boolop("or", P.boolop("and", a, b), P.boolop("and", c, P.unaryop("not", d))), "a and b or c and not d"],
    [P.boolop("and", P.boolop("or", a, b), P.boolop("or", c, d)), "(a or b) and (c or d)"],
    [P.compare("!=", P.unaryop("not", P.compare("==", a, b)), P.compare("<=", c, d)), "(not a == b) != (c <= d)"],
    [P.call(P.call("f", P.name("g")), P.subscript(P.attribute(x, "y"), "z"), P.unaryop("+", 1n), new Uint8Array([0]), true, "it's"),
      "f(g)(x.y['z'], +1, b'\\x00', True, \"it's\")"],
    [P.import_("a", P.import_("b", P.importfrom("d", "e", P.importfrom("d", "f", 1n, "g")), "c")),
      "import a\nimport b as c\nfrom d import e\nfrom d import f as g\n1"],
  ] as [F.Node, string][]) {
    assert(P.render(built) === source, P.render(built));
  }
  assert(!("parse" in P)); // reading Python source needs Python's parser: the Python implementation's only
  assert(same([NaN, Infinity, -Infinity, -2n].map((v) => P.render(P.constant(v))), [
    "float('nan')", "float('inf')", "-float('inf')", "-2"]));
  assert(P.render(P.binop("**", -2n, 2n)) === "(-2) ** 2" && P.render(P.unaryop("-", P.binop("**", 2n, 2n))) === "-2 ** 2");
  assert(P.render(P.call("np.ma.getmaskarray", P.name("x"))) === "np.ma.getmaskarray(x)"); // a dotted function
  assert(P.render(P.attribute(P.binop("+", 1n, 2n), "real")) === "(1 + 2).real");
  raises(ValueError, () => P.render(P.binop("+", 1n, P.import_("math", 2n))), "an import can only enclose the whole expression");
  assert(same(P.import_("numpy.ma", 1n).binds(), ["numpy"]) && same(P.importfrom("m", "n", 1n, "k").binds(), ["k"]));
  assert(same(P.import_("not a module", 1n).validate(), ["an import's module must be a dotted name, got 'not a module'"]));
  assert(same(P.import_("m", 1n, "a b").validate(), ["an import's alias must be an identifier, got 'a b'"]));
  assert(same(P.importfrom("m.", "x y", 1n).validate(), ["an importfrom's module must be a dotted name, got 'm.'",
    "an importfrom's name must be an identifier, got 'x y'"]));
  assert(same(P.importfrom("m", "n", 1n, "").validate(), ["an importfrom needs an alias"])); // empty, not absent
  assert(same(P.let_("a b", 1n, 2n).validate(), ["a let's name must be an identifier, got 'a b'"]));
  assert(same(P.attribute(P.name("x"), "1").validate({ bound: ["x"] }), ["an attribute must be an identifier, got '1'"]));
  assert(P.importfrom("m", "n", P.name("n")).validate().length === 0
    && same(new (kindOf(P.DIALECT, "import"))(7n, null, P.constant(1n)).validate(), ["an import's module must be a str, got int"]));
  assert(same(P.binop("@", 1n, 2n).validate({ core: true }), ["'@' is not a core operation"]));
  assert(P.DIALECT.infer(P.binop("+", 1n, 2.5)) === PD.Float && P.DIALECT.infer(P.binop("+", true, 1n)) === PD.Int);
  assert(P.DIALECT.infer(P.binop("+", "a", "b")) === PD.Str && P.DIALECT.infer(P.unaryop("-", 2.0)) === PD.Float);
  assert(P.DIALECT.infer(P.binop("**", 2n, 3n)).name() === "int | float" && P.DIALECT.infer(P.unaryop("not", 1n)) === PD.Bool);
  assert(P.DIALECT.infer(P.compare("<", 1n, 2.5)) === PD.Bool && P.DIALECT.infer(P.attribute(P.name("x"), "a"), { x: D.Anything }) === D.Anything);
  raises(TypeError, () => P.DIALECT.infer(P.compare("<", "a", 1n)), "< cannot take (str, int)");
  assert(PD.of(new Uint8Array()) === PD.Bytes && PD.Numeric.includes(PD.Bool));
}

## DIA-02 · Python evaluates by Python's rules, in a scope that allowlists what an expression may reach

In [ ]:
{
  const ev = (expression: unknown, variables: Record<string, unknown> = {}, options: any = {}) =>
    PV.OfAny(expression, new PV.Scope(variables, options));
  const [a, t] = [P.name("a"), P.name("t")];
  assert(ev(P.boolop("or", 0n, "x")) === "x" && ev(P.boolop("and", "", 1n)) === "" && ev(P.boolop("and", 1n, 2n)) === 2n); // an operand, not a bool
  assert(ev(P.compare("==", 1n, 1.0)) === true && ev(P.binop("+", true, 1n)) === 2n);
  assert(ev(P.binop("-", P.binop("%", P.binop("//", 7n, 2n), 3n), P.binop("/", P.binop("**", 2n, 2n), 4n))) === -1.0);
  assert(ev(P.ifexp(P.compare(">", a, 1n), a, P.name("missing")), { a: 2n }) === 2n
    && ev(P.let_("a", 3n, P.binop("*", a, a))) === 9n);
  assert(ev(P.attribute(t, "age"), { t: ann }) === 30n && ev(P.call("hasattr", t, "email"), { t: bob }) === false); // properties are attributes
  assert(ev(P.call("getattr", t, "name"), { t: ann }) === "Ann" && ev(P.subscript(t, "x"), { t: { x: 1n } }) === 1n);
  assert(ev(P.call("hasattr", t, "_secret"), { t: ann }) === false);
  assert(ev(P.binop("+", P.binop("+", P.call("abs", P.unaryop("-", 2n)), P.call("len", "abc")), P.call("max", 1n, 2n))) === 7n
    && ev(P.call("f", 2n), { f: (v: bigint) => v + 1n }) === 3n);
  raises(AttributeError, () => ev(P.attribute(t, "email"), { t: bob }), "'Person' object has no attribute 'email'");
  raises(AttributeError, () => ev(P.attribute(t, "__class__"), { t: ann }), "attribute '__class__' is private");
  raises(NameError, () => ev(P.name("x")), "name 'x' is not defined");
  raises(NameError, () => ev(P.call("len", "a"), {}, { builtins: {} }), "name 'len' is not defined");
  function shout(value: string): string {
    return value.toUpperCase();
  }
  raises(TypeError, () => ev(P.call(P.subscript(t, "f"), "a"), { t: { f: shout } }),
    "calling 'shout' is not allowed by the scope"); // only what the scope provides, not what values hold
  raises(TypeError, () => ev(P.call(t), { t: 3n }), "calling 'int' is not allowed by the scope");

  // NumPy is Python's; a module of the scope stands in for it here.
  const masked = { masked: true };
  const numpy = { numpy: new PV.Module("numpy", {
    ma: new PV.Module("numpy.ma", { masked }), pi: Math.PI, _core: new PV.Module("numpy._core", {}) }) };
  assert(ev(P.import_("numpy.ma", P.attribute(P.attribute(P.name("numpy"), "ma"), "masked")), {}, { modules: numpy }) === masked); // binds numpy
  assert(ev(P.import_("numpy.ma", P.attribute(P.name("ma"), "masked"), "ma"), {}, { modules: numpy }) === masked
    && ev(P.import_("numpy", P.attribute(P.name("numpy"), "pi")), {}, { modules: numpy }) === Math.PI);
  raises(ImportError, () => ev(P.import_("numpy", 1n, "np")), "import of 'numpy' is not allowed by the scope"); // nothing is imported unless the scope allows it
  raises(ImportError, () => ev(P.import_("os", P.call("os.getcwd")), {}, { modules: numpy }), "import of 'os' is not allowed by the scope");
  raises(ImportError, () => ev(P.import_("numpy.nope", 1n), {}, { modules: numpy }), "No module named 'numpy.nope'");
  raises(ImportError, () => ev(P.import_("numpy._core", 1n), {}, { modules: numpy }), "No module named 'numpy._core'");
  raises(ImportError, () => ev(P.importfrom("numpy", "nope", 1n), {}, { modules: numpy }), "cannot import name 'nope' from 'numpy'");
  raises(ImportError, () => ev(P.importfrom("numpy", "_core", 1n), {}, { modules: numpy }), "cannot import name '_core' from 'numpy'");
  assert(PV.OfAny(P.binop("+", P.name("x"), 1n), { x: 1n }) === 2n); // an object is the variables of a scope

  // Python's rules, case by case. The Python suite runs this table against Python itself; here it runs against the
  // model of them.
  const n = P.name;
  const bytes = (...values: number[]) => new Uint8Array(values);
  const modules = { math: new PV.Module("math", { floor: (v: number) => BigInt(Math.floor(v)) }), ...numpy };
  const VALUES: [F.Node, Record<string, unknown>, unknown][] = [ // (expression, variables, value): the value and its type
    [P.unaryop("not", n("v")), { v: null }, true], [P.unaryop("not", 0.0), {}, true], [P.unaryop("not", ""), {}, true],
    [P.unaryop("not", bytes()), {}, true], [P.unaryop("not", n("v")), { v: [] }, true], [P.unaryop("not", n("v")), { v: {} }, true],
    [P.unaryop("not", NaN), {}, false], [P.unaryop("not", n("v")), { v: { k: 1n } }, false],
    [P.compare("<", 1n, 1.5), {}, true], [P.compare(">", 2n, 1.5), {}, true], [P.compare("==", 1n, 1.5), {}, false],
    [P.compare("==", P.binop("+", P.binop("**", 2n, 60n), 1n), P.call("float", P.binop("**", 2n, 60n))), {}, false],
    [P.compare("<", 1n, Infinity), {}, true], [P.compare(">", 1n, -Infinity), {}, true], [P.compare("<", 1n, NaN), {}, false],
    [P.compare("<", 1.5, 2n), {}, true], [P.compare("==", NaN, NaN), {}, false], [P.compare("==", 3n, 3.0), {}, true],
    [P.compare("<", 3n, 3.5), {}, true], [P.compare("<", 2.5, 1.5), {}, false], [P.compare("<", true, 2n), {}, true],
    [P.compare("<", "a", "b"), {}, true], [P.compare("<", bytes(97), bytes(98)), {}, true], [P.compare("==", bytes(97), bytes(97)), {}, true],
    [P.compare("!=", "a", 1n), {}, true],
    [P.binop("+", "ab", "c"), {}, "abc"], [P.binop("+", bytes(97), bytes(98)), {}, bytes(97, 98)], [P.binop("*", "ab", 2n), {}, "abab"],
    [P.binop("*", 2n, bytes(120)), {}, bytes(120, 120)], [P.binop("*", "ab", -1n), {}, ""], [P.binop("//", -7n, 2n), {}, -4n],
    [P.binop("%", -7n, 2n), {}, 1n], [P.binop("%", 7n, -2n), {}, -1n], [P.binop("%", -7.5, 2n), {}, 0.5], [P.binop("%", 7.5, 2n), {}, 1.5],
    [P.binop("//", 7.5, 2n), {}, 3.0], [P.binop("**", 2n, -1n), {}, 0.5], [P.binop("**", 2.0, 3n), {}, 8.0],
    [P.binop("+", 1.5, 1n), {}, 2.5], [P.binop("*", 1.5, 2n), {}, 3.0], [P.binop("-", 3n, 1.5), {}, 1.5], [P.binop("/", 1n, 2n), {}, 0.5],
    [P.unaryop("-", true), {}, -1n], [P.unaryop("+", true), {}, 1n], [P.unaryop("-", 1.5), {}, -1.5], [P.unaryop("+", 1.5), {}, 1.5],
    [P.call("abs", true), {}, 1n], [P.call("abs", -1.5), {}, 1.5], [P.call("bool", ""), {}, false], [P.call("float", 1n), {}, 1.0],
    [P.call("float", " 1e3 "), {}, 1000.0], [P.call("float", "inf"), {}, Infinity], [P.call("float", "-inf"), {}, -Infinity],
    [P.call("str", P.call("float", "nan")), {}, "nan"], [P.call("int", " 42 "), {}, 42n], [P.call("int", 2.7), {}, 2n],
    [P.call("int", -2.7), {}, -2n], [P.call("int", true), {}, 1n], [P.call("len", bytes(97, 98)), {}, 2n],
    [P.call("len", n("v")), { v: [1n, 2n] }, 2n], [P.call("len", n("v")), { v: { a: 1n } }, 1n], [P.call("len", "héllo"), {}, 5n],
    [P.call("max", 1n, 2.5), {}, 2.5], [P.call("min", 3n, 1n, 2n), {}, 1n], [P.call("round", 2.5), {}, 2n],
    [P.call("round", 3.5), {}, 4n], [P.call("round", -0.5), {}, 0n], [P.call("round", 7n), {}, 7n], [P.call("str", 1.0), {}, "1.0"],
    [P.call("str", bytes(120)), {}, "b'x'"], [P.call("str", n("v")), { v: null }, "None"], [P.call("str", true), {}, "True"],
    [P.call("str", 7n), {}, "7"], [P.call("str", "s"), {}, "s"], [P.import_("math", P.call("math.floor", 2.5)), {}, 2n],
    [P.unaryop("-", false), {}, 0n], [P.unaryop("not", n("v")), { v: ann }, false], [P.compare("==", 1.5, 1.5), {}, true],
    [P.binop("/", 1.5, 2n), {}, 0.75],
  ];
  const pyEqual = (a: unknown, b: unknown) => Repr.typeName(a) === Repr.typeName(b)
    && (a instanceof Uint8Array ? F.sameNative(a, b) : a === b);
  for (const [expression, variables, expected] of VALUES) {
    const value = PV.OfAny(expression, new PV.Scope(variables, { modules }));
    assert(pyEqual(value, expected), `${P.render(expression)}: ${Repr.repr(value)}, not ${Repr.repr(expected)}`);
  }
  const ERRORS: [F.Node, Record<string, unknown>, Function, string][] = [ // (expression, variables, error, message)
    [P.compare("<", "a", 1n), {}, TypeError, "'<' not supported between instances of 'str' and 'int'"],
    [P.binop("*", "a", 1.5), {}, TypeError, "can't multiply sequence by non-int of type 'float'"],
    [P.binop("-", "a", 1n), {}, TypeError, "unsupported operand type(s) for -: 'str' and 'int'"],
    [P.binop("/", 1n, 0n), {}, ZeroDivisionError, "division by zero"],
    [P.binop("//", 1.5, 0n), {}, ZeroDivisionError, "division by zero"],
    [P.binop("%", 1n, 0n), {}, ZeroDivisionError, "division by zero"],
    [P.binop("**", 0.0, -1n), {}, ZeroDivisionError, "zero to a negative power"],
    [P.unaryop("-", "a"), {}, TypeError, "bad operand type for unary -: 'str'"],
    [P.call("abs", "a"), {}, TypeError, "bad operand type for abs(): 'str'"],
    [P.call("float", "x"), {}, ValueError, "could not convert string to float: 'x'"],
    [P.call("int", "x"), {}, ValueError, "invalid literal for int() with base 10: 'x'"],
    [P.call("int", NaN), {}, ValueError, "cannot convert float NaN to integer"],
    [P.call("int", Infinity), {}, OverflowError, "cannot convert float infinity to integer"],
    [P.call("len", 1n), {}, TypeError, "object of type 'int' has no len()"],
    [P.call("max", 1n, "a"), {}, TypeError, "'>' not supported between instances of 'str' and 'int'"],
    [P.call("round", "x"), {}, TypeError, "type str doesn't define __round__ method"],
    [P.call("float", n("v")), { v: null }, TypeError, "float() argument must be a string or a real number, not 'NoneType'"],
    [P.call("int", n("v")), { v: null }, TypeError,
      "int() argument must be a string, a bytes-like object or a real number, not 'NoneType'"],
    [P.import_("numpy", P.attribute(n("numpy"), "nope")), {}, AttributeError, "module 'numpy' has no attribute 'nope'"],
    [P.subscript(n("t"), "y"), { t: { x: 1n } }, KeyError, "'y'"],
    [P.subscript(n("t"), "x"), { t: 3n }, TypeError, "'int' object is not subscriptable"],
    [P.call(P.subscript(n("t"), "f")), { t: { f: [() => 1n][0] } }, TypeError, "calling '<lambda>' is not allowed by the scope"],
  ];
  for (const [expression, variables, error, message] of ERRORS) {
    raises(error, () => PV.OfAny(expression, new PV.Scope(variables, { modules })), message);
  }
  assert(PV.OfAny(P.importfrom("numpy.ma", "masked", n("masked")), new PV.Scope({}, { modules })) === masked);
  // A mapping is a Map or a plain object.
  assert(ev(P.subscript(t, "x"), { t: new Map([["x", 1n]]) }) === 1n && ev(P.unaryop("not", t), { t: new Map() }) === true);
  raises(KeyError, () => ev(P.subscript(t, "y"), { t: new Map() }), "'y'");
  assert(ev(P.call("len", t), { t: new Map([["x", 1n]]) }) === 1n);
}

## DIA-03 · Matlab: operators, fields and `isfield`, rendered as MATLAB source

In [ ]:
const s = M.identifier("s");
const matlabRule = M.binary("&&", M.binary(">=", M.field(s, "age"), 18n), M.call("isfield", s, "email"));
{
  assert(M.render(matlabRule) === 's.age >= 18 && isfield(s, "email")');
  assert(M.render(M.binary("-", 1n, M.binary("-", 2n, 3n))) === "1 - (2 - 3)"
    && M.render(M.binary("-", M.binary("-", 1n, 2n), 3n)) === "1 - 2 - 3");
  assert(M.render(M.binary(".*", M.binary("+", 1n, 2n), M.unary("-", M.unary("~", s)))) === "(1 + 2) .* -~s");
  assert(M.render(M.unary("-", M.binary("+", 1n, 2n))) === "-(1 + 2)" && M.render(M.unary("~", -1n)) === "~-1");
  assert(M.render(M.binary("||", M.binary("&&", true, false), M.binary("&&", 1n, 0n))) === "true && false || 1 && 0");
  assert(same([NaN, Infinity, -Infinity, -2.5, 'say "hi"', 7n].map((v) => M.render(M.constant(v))), [
    "NaN", "Inf", "-Inf", "-2.5", '"say ""hi"""', "7"]));
  assert(M.render(M.field(M.field(s, "a"), "b")) === "s.a.b");
  assert(M.DIALECT.infer(matlabRule, { s: MD.Struct }) === MD.Logical);
  assert(M.DIALECT.infer(M.binary("+", "a", 1n)) === MD.String && M.DIALECT.infer(M.binary("+", true, 1n)) === MD.Double);
  assert(MD.Double.contains(1n) && MD.Struct.contains({}) && !MD.Struct.contains("s") && MD.of(1n) === MD.Double);
  assert(same(M.binary("^", 1n, 2n).validate({ core: true }), ["'^' is not a core operation"]));
}

## DIA-04 · Matlab evaluates two-valued, converts numbers and logicals, and has no unknown

In [ ]:
{
  assert(MV.OfAny(matlabRule, { s: ann }) === true && MV.OfAny(matlabRule, { s: bob }) === false);
  assert(MV.OfAny(matlabRule, { s: { age: 20n, email: "x" } }) === true); // a mapping is a struct
  assert(MV.OfAny(M.field(s, "age"), { s: ann }) === 30 && typeof MV.OfAny(18n) === "number"); // numbers are doubles
  assert(MV.OfAny(M.binary("==", 1n, true)) === true && MV.OfAny(M.binary("+", true, 1n)) === 2);
  assert(MV.OfAny(M.binary("<", "apple", "banana")) === true && MV.OfAny(M.binary("==", "5", 5n)) === true);
  assert(MV.OfAny(M.binary("==", "true", true)) === true && MV.OfAny(M.binary("~=", "1.5", 1.5)) === false);
  assert(MV.OfAny(M.binary("+", "a", 1n)) === "a1" && MV.OfAny(M.binary("+", "x", 0.125)) === "x0.125");
  assert(same([NaN, Infinity, -Infinity, 12345.6, 1.5e-7, 123456.7, true, false].map((v) => MV.OfAny(M.binary("+", "x", v))), [
    "xNaN", "xInf", "x-Inf", "x12346", "x1.5e-07", "x1.2346e+05", "xtrue", "xfalse"])); // numbers as MATLAB writes them
  assert(MV.OfAny(matlabRule, { s: new Map<string, unknown>([["age", 20n], ["email", "x"]]) }) === true); // a Map is a struct too
  assert(same(["~=", "<=", ">", ">=", "-", ".*"].map((op) => MV.OfAny(M.binary(op, 2n, 3n))), [true, true, false, false, -1, 6]));
  assert(MV.OfAny(M.binary("&&", false, M.identifier("missing"))) === false); // short-circuits
  assert(MV.OfAny(M.binary("||", 2n, M.identifier("missing"))) === true && MV.OfAny(M.binary("||", 0n, 0n)) === false);
  assert(MV.OfAny(M.unary("~", 0n)) === true && MV.OfAny(M.unary("-", true)) === -1);
  assert(MV.OfAny(M.call("isfield", 3n, "x")) === false && MV.OfAny(M.call("isfield", s, 1n), { s: ann }) === false);
  raises(KeyError, () => MV.OfAny(M.field(s, "email"), { s: bob }), 'Unrecognized field name "email".');
  raises(TypeError, () => MV.OfAny(M.field(s, "age"), { s: 3n }), "Dot indexing is not supported for variables of this type.");
  raises(NameError, () => MV.OfAny(M.identifier("missing")), "Unrecognized function or variable 'missing'.");
  raises(TypeError, () => MV.OfAny(M.binary("&&", "yes", true)), "Operands to the && operator must be convertible to logical scalar values.");
  raises(ValueError, () => MV.OfAny(M.unary("~", NaN)), "NaN's cannot be converted to logicals.");
  raises(TypeError, () => MV.OfAny(M.binary("-", "a", 1n)), "Operator '-' is not supported for operands of type 'string' and 'double'.");
  raises(TypeError, () => MV.OfAny(M.binary(".*", s, true), { s: ann }), "Operator '.*' is not supported for operands of type 'struct' and 'logical'.");
  raises(TypeError, () => MV.OfAny(M.unary("-", "a")), "Operator '-' is not supported for operands of type 'string'.");
}

## DIA-05 · Matlab resolves functions from imports, the path and packages that the scope provides

In [ ]:
{
  const geo = { dist: (a: number, b: number) => (a * a + b * b) ** 0.5, area: (r: number) => 3 * r * r };
  const scope = new MV.Scope({ r: 2n }, { functions: { twice: (v: number) => 2 * v },
    packages: { geo, "geo.more": { area: () => 0n } } });
  const wild = M.import_("geo.*", M.binary("+", M.call("dist", 3n, 4n), M.call("area", M.identifier("r"))));
  assert(M.render(wild) === "import geo.*\ndist(3, 4) + area(r)" && wild.validate({ bound: ["r"] }).length === 0);
  assert(same(wild.validate({ bound: ["r"], core: true }), ["body: left: 'dist' is not a core operation",
    "body: right: 'area' is not a core operation"]));
  assert(MV.OfAny(wild, scope) === 17);
  assert(MV.OfAny(M.import_("geo.dist", M.call("dist", 6n, 8n)), scope) === 10);
  assert(MV.OfAny(M.call("geo.more.area", 1n), scope) === 0 && MV.OfAny(M.call("twice", 4n), scope) === 8); // an int result is a double
  assert(MV.OfAny(M.import_("geo.*", M.import_("geo.more.area", M.call("area", 5n))), scope) === 0); // innermost wins
  raises(NameError, () => MV.OfAny(M.call("dist", 3n, 4n), scope), "Unrecognized function or variable 'dist'."); // not imported
  raises(ImportError, () => MV.OfAny(M.import_("os.system", 1n), scope), "Import argument 'os.system' cannot be found or cannot be imported.");
  raises(ImportError, () => MV.OfAny(M.import_("geo.nope", 1n), scope), "Import argument 'geo.nope' cannot be found or cannot be imported.");
  raises(ImportError, () => MV.OfAny(M.import_("nope.*", 1n), scope), "Import argument 'nope.*' cannot be found or cannot be imported.");
  raises(ValueError, () => M.render(M.binary("+", 1n, M.import_("geo.*", 2n))), "an import can only enclose the whole expression");
  assert(same(M.import_("geo", 1n).validate(), ["an import's name must be pkg.name or pkg.*, got 'geo'"]));
  assert(same(M.import_("geo.*.x", 1n).validate(), ["an import's name must be pkg.name or pkg.*, got 'geo.*.x'"]));
  assert(same(new (kindOf(M.DIALECT, "import"))(7n, M.constant(1n)).validate(), ["an import's name must be a str, got int"]));
}

## DIA-06 · Excel: formulas with `LET`, functions, operators and fields

In [ ]:
const r = X.name("r");
const excelRule = X.let_("age", X.field(r, "age"), X.function("AND", X.infix(">=", X.name("age"), 18n),
  X.function("NOT", X.function("ISERROR", X.field(r, "email")))));
{
  assert(X.render(excelRule) === "=LET(age, r.age, AND(age >= 18, NOT(ISERROR(r.email))))");
  assert(X.render(X.infix("*", X.infix("+", 1n, 2n), X.prefix("-", X.infix("-", 3n, 4n)))) === "=(1 + 2) * -(3 - 4)");
  assert(X.render(X.infix("=", X.infix("-", 1n, X.infix("-", 2n, 3n)), -1n)) === "=1 - (2 - 3) = -1");
  assert(X.render(X.prefix("-", X.field(r, "first name"))) === "=-r.[first name]");
  assert(same([true, false, 'say "hi"', Infinity, 2.5, 3n].map((v) => X.render(X.constant(v))), [
    "=TRUE", "=FALSE", '="say ""hi"""', "=#NUM!", "=2.5", "=3"]));
  assert(excelRule.validate({ bound: ["r"], core: true }).length === 0
    && same(X.function("VLOOKUP", 1n, 2n).validate({ core: true }), ["'VLOOKUP' is not a core operation"]));
  assert(X.DIALECT.infer(excelRule, { r: XD.Record }) === XD.Logical && X.DIALECT.infer(X.infix("+", "1", true)) === XD.Number);
  assert(XD.of(1n) === XD.Number && XD.Errors.contains(XD.Error.of("#N/A")) && String(XD.Error.of("#N/A")) === "#N/A");
  assert(XD.Record.contains(ann) && !XD.Record.contains(1n));
}

## DIA-07 · Excel coerces, orders values of every type, and computes with errors

In [ ]:
{
  const Error = XV.Error;
  assert(XV.OfAny(excelRule, { r: ann }) === true && XV.OfAny(excelRule, { r: bob }) === false); // ISERROR finds the #FIELD!
  assert(XV.OfAny(X.field(r, "email"), { r: bob }) === Error.of("#FIELD!") && XV.OfAny(X.name("nope")) === Error.of("#NAME?"));
  assert(XV.OfAny(X.field(r, "age"), { r: { age: 5n } }) === 5 && XV.OfAny(X.field(r, "x"), { r: 3n }) === Error.of("#VALUE!"));
  assert(XV.OfAny(X.field(X.field(r, "a"), "b"), { r: {} }) === Error.of("#FIELD!")); // errors propagate
  assert(XV.OfAny(X.infix("+", "1", true)) === 2 && XV.OfAny(X.infix("*", "x", 1n)) === Error.of("#VALUE!")); // coercion
  assert(XV.OfAny(X.infix("-", 1n, X.infix("+", X.name("u"), 1n))) === Error.of("#NAME?"));
  assert(XV.OfAny(X.prefix("-", "2")) === -2 && XV.OfAny(X.prefix("-", X.name("u"))) === Error.of("#NAME?"));
  assert(XV.OfAny(X.infix(">", "a", 1n)) === true && XV.OfAny(X.infix(">", true, "z")) === true); // number < text < logical
  assert(XV.OfAny(X.infix("=", "a", "A")) === true && XV.OfAny(X.infix("<>", 1n, "1")) === true); // case; no coercion
  assert(same(["<", "<=", ">=", "-"].map((op) => XV.OfAny(X.infix(op, 2n, 3n))), [true, true, false, -1]));
  assert(XV.OfAny(X.infix("=", 2n, 2n)) === true && XV.OfAny(X.infix("+", false, 1n)) === 1);
  assert(XV.OfAny(X.infix("+", "inf", 1n)) === Error.of("#VALUE!") && XV.OfAny(X.infix("+", " 1e3 ", 1n)) === 1001); // decimal text
  assert(XV.OfAny(X.infix("+", r, 1n), { r: {} }) === Error.of("#VALUE!"));
  assert(XV.OfAny(X.field(r, "age"), { r: new Map([["age", 5n]]) }) === 5); // a Map is a record too
  assert(XV.OfAny(X.infix("=", X.name("u"), 1n)) === Error.of("#NAME?") && XV.OfAny(X.infix("=", r, 1n), { r: {} }) === Error.of("#VALUE!"));
  assert(XV.OfAny(X.function("AND", 1n, true)) === true && XV.OfAny(X.function("OR", 0n, false)) === false);
  assert(XV.OfAny(X.function("AND", false, X.name("u"))) === Error.of("#NAME?")); // AND evaluates every argument
  assert(XV.OfAny(X.function("OR", "yes", true)) === Error.of("#VALUE!") && XV.OfAny(X.function("NOT", 0n)) === true);
  assert(XV.OfAny(X.function("NOT", X.name("u"))) === Error.of("#NAME?"));
  assert(XV.OfAny(X.function("IF", true, 1n, X.name("u"))) === 1 && XV.OfAny(X.function("IF", 0n, 1n, "no")) === "no");
  assert(XV.OfAny(X.function("IF", X.name("u"), 1n, 2n)) === Error.of("#NAME?")); // IF evaluates only the branch it takes
  assert(XV.OfAny(X.function("ISERROR", 1n)) === false && XV.OfAny(X.let_("x", 2n, X.infix("*", X.name("x"), 3n))) === 6);
}

## DIA-08 · Excel reads cells from workbooks, and add-in functions from the workbook's add-ins

In [ ]:
{
  const Error = XV.Error;
  const rates = new XV.Workbook({}, { Rates: { B2: 0.5 } }, { name: "Rates.xlsx" });
  const book = new XV.Workbook({ base: 10n }, { Sheet1: { a1: 2n, $A$2: 3n }, "My Sheet": { C3: "x" } },
    { books: { "Rates.xlsx": rates }, add_ins: { double: (v: number) => v * 2 } });
  const total = X.infix("+", X.infix("*", X.cell("A1"), X.cell("B2", "Rates", "Rates.xlsx")), X.function("DOUBLE", X.name("base")));
  assert(X.render(total) === "=A1 * [Rates.xlsx]Rates!B2 + DOUBLE(base)" && XV.OfAny(total, book) === 21);
  assert(total.validate({ bound: ["base"] }).length === 0
    && same(total.validate({ bound: ["base"], core: true }), ["right: 'DOUBLE' is not a core operation"]));
  assert(X.render(X.cell("C3", "My Sheet")) === "='My Sheet'!C3" && XV.OfAny(X.cell("C3", "My Sheet"), book) === "x");
  assert(X.render(X.cell("A1", "Bob's", "My Book.xlsx")) === "='[My Book.xlsx]Bob''s'!A1");
  assert(XV.OfAny(X.cell("$a$2"), book) === 3 && XV.OfAny(X.cell("Z9"), book) === 0); // an empty cell is 0
  assert(XV.OfAny(X.cell("A1", "Sheet1", "Book1"), book) === 2); // this book, named
  assert(XV.OfAny(X.cell("A1", "Nope"), book) === Error.of("#REF!") && XV.OfAny(X.cell("A1", "S", "Other.xlsx"), book) === Error.of("#REF!"));
  assert(XV.OfAny(X.function("NOPE", 1n), book) === Error.of("#NAME?")
    && XV.OfAny(X.function("double", X.name("u")), book) === Error.of("#NAME?")); // add-ins ignore case; errors in their arguments propagate
  assert(XV.OfAny(X.cell("A1"), new XV.Workbook()) === Error.of("#REF!")); // a workbook has no sheet until given one
  assert(XV.OfAny(X.name("x"), { x: 1n }) === 1 && new XV.Workbook().sheet === "Sheet1");
  raises(ValueError, () => new XV.Workbook({}, { Sheet1: { total: 1n } }), "not a cell address: 'total'");
  assert(same(X.cell("A0").validate(), ["a cell's address must be a column and a row, like A1, got 'A0'"]));
  assert(same(X.cell("A1", null, "B.xlsx").validate(), ["a cell in another book needs a sheet"]));
  assert(X.DIALECT.infer(X.cell("A1")) === D.Anything && X.address("$b$12") === "B12" && X.address("B") === null);
}

## DIA-09 · Latex: notation in math mode, rendered and checked, with no evaluator

In [ ]:
{
  const [L, LD] = [Latex.Expressions, Latex.Domains];
  const [a, self] = [L.symbol("a"), L.symbol("this")];
  const notation = L.where("a", L.member(self, "age"), L.binary("\\land", L.binary("\\geq", a, 18n), L.binary(
    "\\lor", L.unary("\\lnot", L.function("has", self, "email")), L.binary(">", L.frac(a, 2n), 1.5))));
  assert(L.render(notation) === "a \\geq 18 \\land (\\lnot \\operatorname{has}(\\mathit{this}, \\text{email}) \\lor \\frac{a}{2} > 1.5) "
    + "\\quad \\text{where } a = \\mathit{this}.\\mathit{age}");
  assert(L.render(L.binary("\\cdot", L.binary("+", 1n, 2n), L.unary("-", L.binary("-", 3n, -4n)))) === "(1 + 2) \\cdot -(3 - -4)");
  assert(L.render(L.binary("\\implies", L.binary("\\implies", 1n, 2n), L.binary("\\implies", 3n, 4n)))
    === "(1 \\implies 2) \\implies 3 \\implies 4"); // right-associative
  assert(L.render(L.binary("<", L.binary("<", 1n, 2n), 3n)) === "(1 < 2) < 3"); // comparisons do not chain
  assert(L.render(L.member(L.binary("+", self, 1n), "x_y")) === "(\\mathit{this} + 1).\\mathit{x\\_y}");
  assert(L.render(L.where("b", 1n, L.where("a", 2n, a))) === "(a \\quad \\text{where } a = 2) \\quad \\text{where } b = 1");
  assert(same([NaN, Infinity, -Infinity, 1e-7, 1e16, -2n, true, false, "a_b {c}"].map((v) => L.render(L.constant(v))), [
    "\\mathrm{NaN}", "\\infty", "-\\infty", "1 \\times 10^{-7}", "1 \\times 10^{16}", "-2", "\\mathrm{true}",
    "\\mathrm{false}", "\\text{a\\_b \\{c\\}}"]));
  assert(L.render(L.constant("\\$&#%~^")) === "\\text{\\textbackslash{}\\$\\&\\#\\%\\textasciitilde{}\\textasciicircum{}}");
  assert(L.render(L.binary("\\cdot", L.constant(1e16), 2n)) === "1 \\times 10^{16} \\cdot 2");
  assert(notation.validate({ bound: ["this"], core: true }).length === 0
    && same(L.binary("\\times", 1n, 2n).validate({ core: true }), ["'\\\\times' is not a core operation"]));
  assert(L.function("max", 1n, 2n).validate().length === 0 && same(L.function("has", 1n).validate(), ["has takes 2 arguments, got 1"]));
  assert(L.DIALECT.infer(notation, { this: D.Anything }) === LD.Truth && L.DIALECT.infer(L.frac(1n, 2.5)) === LD.Number);
  raises(TypeError, () => L.DIALECT.infer(L.binary("\\land", 1n, true)), "\\land cannot take (number, truth)");
  assert(LD.of("x") === LD.Text && !("Evaluators" in Latex)); // notation is not evaluated
}